# Is the difference real, and is it big?

**Module 1 · Session 03, part 2**

Genres differ in energy. A curation team would ask two things about that, and they are not the
same question.

Is the difference real, or could it be chance? A significance test answers that, and on a
table this size it is nearly free.

Is it big enough to do anything about? No test answers that. That half is the reason this
notebook exists.

## Which test

You will meet three of these today. Choosing between them is the easy part, and it comes down
to what kind of columns you have.

| Your question | The test |
|---|---|
| Do two groups differ on a number? | t-test |
| Do three or more groups differ on a number? | one-way ANOVA |
| Are two categorical columns related? | chi-square |
| Do two numbers move together? | correlation, which we did in part 1 |

Every one of them answers "could this be chance". None of them answers "is this big". Section
4 is where that second number comes from.


In [ ]:
# Setup. Same file as last week. Nothing else to install.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"
local = [Path("data/M1_2026/spotify_songs.csv"), Path("../data/M1_2026/spotify_songs.csv"),
         Path("ds-master/data/M1_2026/spotify_songs.csv"), Path("../ds-master/data/M1_2026/spotify_songs.csv")]

path = next((p for p in local if p.exists()), None)
songs = pd.read_csv(path if path is not None else URL)
songs = songs.rename(columns={"track_name": "title", "track_artist": "artist",
                             "track_popularity": "popularity", "playlist_genre": "genre"})

# One row = one track on one playlist, which is the grain we settled on last week.
# One row per song instead:
tracks = songs.drop_duplicates("track_id")

print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


## 1. Look first

Two genres, one column, before any test.


In [ ]:
two = songs[songs["genre"].isin(["edm", "rock"])]
display(two.groupby("genre")["energy"].agg(rows="size", mean="mean", sd="std").round(3))


In [ ]:
two.boxplot(column="energy", by="genre", figsize=(7, 4), grid=False)
plt.suptitle("")
plt.title("Energy: edm against rock")
plt.show()


Means of about 0.80 and 0.73, so a gap of roughly 0.07 on a scale from 0 to 1, and the boxes
overlap a lot.

Before running anything, write down your two answers. Is that gap real? Is it big?

## 2. The t-test

A two-sample t-test answers one narrow question: if these two groups really had the same mean,
how surprising would a gap this size be?

It does not answer whether the gap is large. Keep those apart for the rest of the notebook.


In [ ]:
edm = songs.loc[songs["genre"] == "edm", "energy"]
rock = songs.loc[songs["genre"] == "rock", "energy"]

# equal_var=False is Welch's version: it does not assume the groups have the same spread.
result = stats.ttest_ind(edm, rock, equal_var=False)

print(f"rows:       {len(edm):,} edm, {len(rock):,} rock")
print(f"means:      {edm.mean():.3f} and {rock.mean():.3f}")
print(f"difference: {edm.mean() - rock.mean():.3f}")
print(f"p value:    {result.pvalue:.2e}")


A p value around ten to the power of minus ninety-seven. Read literally: if edm and rock
really had identical mean energy, a gap this size in samples this size would be so rare the
number stops meaning anything in everyday terms.

That feels like a strong result. Hold onto the feeling for one cell.

## 3. The same test, on a difference you cannot see


In [ ]:
pop = songs.loc[songs["genre"] == "pop", "energy"]
latin = songs.loc[songs["genre"] == "latin", "energy"]

test = stats.ttest_ind(pop, latin, equal_var=False)
print(f"means:      {pop.mean():.4f} and {latin.mean():.4f}")
print(f"difference: {pop.mean() - latin.mean():.4f}")
print(f"p value:    {test.pvalue:.4f}")
print("Significant at 5 percent?", test.pvalue < 0.05)


A difference of seven thousandths of a point, on an index bounded between 0 and 1. Nothing you
could hear. Nothing anyone would act on.

And it is significant.

So the word has now been earned by a gap of 0.070 and by a gap of 0.007. The p value cannot
tell those apart, because that is not the question it answers.


> **Judgement call.** This is the one to remember. Ask an agent whether two genres differ and you get a correct
test, a correct p value and a verdict. Whether anyone should care was not in the request and
is not in the output. On business-sized data that missing half is usually the whole decision.


## 4. Effect size

The number that separates those two results is how big the gap is, measured against how spread
out the data already was.

Cohen's d is the common one: the difference in means divided by the pooled standard deviation.


In [ ]:
def cohens_d(a, b):
    """Difference in means, in pooled standard deviations."""
    return (a.mean() - b.mean()) / np.sqrt((a.std() ** 2 + b.std() ** 2) / 2)


print(f"edm against rock:  d = {cohens_d(edm, rock):.2f}")
print(f"pop against latin: d = {cohens_d(pop, latin):.2f}")


The rough labels are 0.2 small, 0.5 medium, 0.8 large. They are conventions rather than laws,
and knowing them mostly shows you how far below them real results sit.

edm against rock is about 0.4, a moderate difference, roughly what the box plot suggested.
pop against latin is about 0.05, which is nothing. Same p-value verdict, different worlds.

Report the effect size to someone deciding what to do. Report the p value to someone asking
whether you might be looking at noise.

Put it as a decision and it gets obvious. Suppose the pop-latin result had been the one you
were asked about: mean energy 0.701 against 0.708, significant at 5 percent. What would you
change on the strength of it? Nothing. No playlist gets re-sorted over seven thousandths of a
bounded index. The test was correct and the answer was still worthless.

That gap, between real and worth acting on, is where most bad analysis lives.

## 5. Why sample size did all of this

Everything is significant here because each group has about five thousand rows. Take the
comparison with no real difference and run it on small samples instead.


In [ ]:
p_values = np.array([
    stats.ttest_ind(pop.sample(30, random_state=seed),
                    latin.sample(30, random_state=seed + 999), equal_var=False).pvalue
    for seed in range(200)
])

print(f"median p value:              {np.median(p_values):.3f}")
print(f"significant at 5 percent in: {(p_values < 0.05).mean():.1%} of runs")


On thirty rows a side, the difference that was significant a moment ago shows up about five
percent of the time, which is exactly what you would get from chance alone.

The data never changed. Only the number of rows did. That is the mechanism: a p value is a
statement about a difference **and** your sample size, and with enough rows any difference
that is not exactly zero eventually crosses any threshold you choose.

## 6. More than two groups

Same reasoning, two more tests. The verdict is cheap; the size is the finding.


In [ ]:
groups = [g["energy"].values for _, g in songs.groupby("genre")]
anova = stats.f_oneway(*groups)
print(f"F: {anova.statistic:.0f}, p: {anova.pvalue:.1e}")

# The size that goes with it: how much of the variation sits between genres.
grand_mean = songs["energy"].mean()
between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
total = ((songs["energy"] - grand_mean) ** 2).sum()
print(f"eta squared: {between / total:.3f}")


Genre explains about 14 percent of the variation in energy. So roughly 86 percent of it is
variation **within** genres: two edm tracks usually differ more from each other than the
average edm track differs from the average rock track.

That one sentence is a better summary of this dataset than any p value in the notebook.

## 7. Two categorical columns

`mode` is 1 for major and 0 for minor. Both columns are categories, so the test changes and the
reasoning does not.


In [ ]:
table = pd.crosstab(songs["genre"], songs["mode"])
table.columns = ["minor", "major"]
display(table.div(table.sum(axis=1), axis=0).round(2))   # shares, easier to read than counts


In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print(f"chi-square: {chi2:.0f}, p: {p_value:.1e}")
print(f"Cramer's V: {cramers_v:.3f}")


## Before you go hunting

Run twenty tests on data with nothing in it and about one will come back significant. That is
what a 5 percent threshold means: it is the rate at which you expect to be fooled.

This used to be a slow problem, because running twenty tests took an afternoon. An agent will
run fifty comparisons in one call and hand you the three that cleared 0.05, which is a machine
for generating findings that are not there.

Test the question you arrived with. If you do go looking across every pair of columns, say so,
and treat whatever turns up as something to check against other data rather than as a result.


Significant again, and a Cramer's V of 0.13, which is weak.

But read the share table rather than the single number. Rock is 30 percent minor; the others
sit between 41 and 48 percent. Rock is the story, and the overall V is weak partly because the
other five genres sit close together and dilute it.

A single effect-size number for a six-by-two table is a summary, and summaries hide things.

## 8. How to report it

Never report a p value on its own. Report three things, in this order:

1. the size of the difference, in the units you measured;
2. how many observations it rests on;
3. the p value, last, as the smallest of the three claims.

"Mean energy is 0.80 for edm and 0.73 for rock, across 6,043 and 4,951 rows, a gap of 0.07 and
d = 0.41 (p < 0.001)" is a sentence someone can argue with.

"The difference was significant (p < 0.001)" is not.

## Practice

Compare `danceability` between rap and rock. Report the means, the row counts, Cohen's d and
the p value, then write one sentence saying whether a playlist team should care.


In [ ]:
# Your turn.


## Solution


In [ ]:
rap_d = songs.loc[songs["genre"] == "rap", "danceability"]
rock_d = songs.loc[songs["genre"] == "rock", "danceability"]

print(f"rap:  {len(rap_d):,} rows, mean {rap_d.mean():.3f}")
print(f"rock: {len(rock_d):,} rows, mean {rock_d.mean():.3f}")
print(f"difference: {rap_d.mean() - rock_d.mean():.3f}")
print(f"Cohen's d:  {cohens_d(rap_d, rock_d):.2f}")
print(f"p value:    {stats.ttest_ind(rap_d, rock_d, equal_var=False).pvalue:.1e}")


## One open question

Pick a comparison in this file that you would actually want the answer to. Any two groups, any
column. Run it properly: counts, the size of the difference, the effect size, then the p value.

Then write the single sentence you would send to the curation team, and one sentence saying
what it does not tell them.


In [ ]:
# Your comparison.


Model answer: rap sits near 0.72 and rock near 0.52, a gap of about 0.20 with an effect size
of about 1.4. That is very large by any convention and you can see it in a box plot without a test.
Yes, a playlist team should care.

Notice that this is also the case where the p value adds least. Nothing about the conclusion
depends on it.

## Takeaways

- A p value answers whether a difference exists given your sample size. It says nothing about
  whether the difference is big.
- On business-sized tables almost everything is significant, so effect size carries the finding.
- Report size, then counts, then the p value.
- Genre explains about 14 percent of the variation in energy here. Most variation is within
  genres, not between them.
- Twenty tests on nothing produce about one significant result. Test the question you came
  with.
- All of this describes one 2020 playlist snapshot. It is not evidence about music or listeners
  now.
